# Dataset Statistical Analysis for Approximate Nearest Neighbors (ANN) Search

This notebook provides a comprehensive statistical analysis of the 7 HDF5 datasets in the `../dataset/` directory (repo root). Understanding the distribution, normalization, dimensionality, and neighborhood structure of these datasets is key to designing and optimizing algorithms such as **HNSW** (Hierarchical Navigable Small World) or **LSH** (Locality Sensitive Hashing).

---

In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for figures
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

dataset_dir = '../dataset'
files = sorted([f for f in os.listdir(dataset_dir) if f.endswith('.hdf5')])
print(f"Found {len(files)} HDF5 datasets in the directory.")

## 1. Dataset Dimensions & Basic Metadata

We load basic information for each dataset including the number of training points ($N$), test/query points ($Q$), dimensionality ($D$), and the size on disk.

In [ ]:
metadata = []
for f_name in files:
    f_path = os.path.join(dataset_dir, f_name)
    disk_size_gb = os.path.getsize(f_path) / (1024**3)
    with h5py.File(f_path, 'r') as f:
        n_train = f['train'].shape[0]
        dim = f['train'].shape[1]
        n_test = f['test'].shape[0]
        metadata.append({
            "Dataset": f_name.replace("-public.hdf5", ""),
            "Train Size (N)": n_train,
            "Test Size (Q)": n_test,
            "Dimension (D)": dim,
            "Disk Size (GB)": round(disk_size_gb, 3)
        })

df_meta = pd.DataFrame(metadata)
df_meta.style.format({
    "Train Size (N)": "{:,}",
    "Test Size (Q)": "{:,}",
    "Dimension (D)": "{:,}",
    "Disk Size (GB)": "{:.3f}"
}).set_caption("Overview of ANN Datasets")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Training Set Size (Log Scale due to massive variations)
sns.barplot(data=df_meta, x="Dataset", y="Train Size (N)", ax=axes[0], palette="viridis", hue="Dataset", legend=False)
axes[0].set_title("Training Set Size (N)", fontsize=14, fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha="right")
axes[0].set_yscale("log")
axes[0].set_ylabel("Number of Vectors (Log Scale)")

# Right: Vector Dimension
sns.barplot(data=df_meta, x="Dataset", y="Dimension (D)", ax=axes[1], palette="magma", hue="Dataset", legend=False)
axes[1].set_title("Vector Dimension (D)", fontsize=14, fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha="right")
axes[1].set_ylabel("Dimension")

plt.tight_layout()
plt.show()

## 2. Normalization & Vector Norms Analysis

Many modern embeddings (such as those generated by OpenAI's text models, CLIP, or Nomic) are normalized to have unit L2 norm ($||v||_2 = 1.0$). When vectors are normalized:
- Cosine similarity is equivalent to the dot product ($u \cdot v$).
- Squared Euclidean distance is directly related to cosine similarity by $d^2(u, v) = 2 - 2 \cos(\theta)$.

Let's sample 10,000 vectors from each dataset's training set to analyze the distribution of vector norms.

In [ ]:
norm_stats = []
norm_samples = {}

for f_name in files:
    f_path = os.path.join(dataset_dir, f_name)
    ds_short = f_name.replace("-public.hdf5", "")
    with h5py.File(f_path, 'r') as f:
        train = f['train']
        n_train = train.shape[0]
        
        # Sample up to 10k training points for efficient computation
        np.random.seed(42)
        indices = sorted(np.random.choice(n_train, min(n_train, 10000), replace=False))
        sample = train[indices]
        norms = np.linalg.norm(sample, axis=1)
        
        norm_samples[ds_short] = norms
        norm_stats.append({
            "Dataset": ds_short,
            "Norm Mean": np.mean(norms),
            "Norm Std": np.std(norms),
            "Norm Min": np.min(norms),
            "Norm Max": np.max(norms),
            "Normalized (Unit Norm)?": "Yes" if np.allclose(norms, 1.0, atol=1e-3) else "No"
        })

df_norms = pd.DataFrame(norm_stats)
df_norms.style.format({
    "Norm Mean": "{:.4f}",
    "Norm Std": "{:.4f}",
    "Norm Min": "{:.4f}",
    "Norm Max": "{:.4f}"
}).set_caption("Vector Norm Summary Statistics")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Non-normalized datasets
non_norm_datasets = [d for d in norm_samples if d in ["agnews-mxbai", "celeba-resnet"]]
for d in non_norm_datasets:
    sns.kdeplot(norm_samples[d], label=d, ax=axes[0], fill=True, alpha=0.3)
axes[0].set_title("L2 Norm Distribution (Non-Normalized)", fontsize=13, fontweight='bold')
axes[0].set_xlabel("L2 Norm Value")
axes[0].legend()

# Right: Normalized datasets
norm_datasets = [d for d in norm_samples if d not in ["agnews-mxbai", "celeba-resnet"]]
for d in norm_datasets:
    sns.kdeplot(norm_samples[d], label=d, ax=axes[1], fill=True, alpha=0.3)
axes[1].set_title("L2 Norm Distribution (Normalized to Sphere)", fontsize=13, fontweight='bold')
axes[1].set_xlabel("L2 Norm Value")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Ground Truth Neighbor Distance Profiles

Each dataset includes the ground truth nearest neighbor distances (`distances` dataset). Let's trace the average distance to the nearest neighbors at different ranks (e.g., 1st nearest, 5th, 10th, 20th, 50th, and 100th) to understand spacing and density.

In [ ]:
dist_profiles = {}
ranks = [0, 4, 9, 19, 49, 99]
ranks_label = [1, 5, 10, 20, 50, 100]

for f_name in files:
    f_path = os.path.join(dataset_dir, f_name)
    ds_short = f_name.replace("-public.hdf5", "")
    with h5py.File(f_path, 'r') as f:
        distances = f['distances'][:]
        mean_dists = distances[:, ranks].mean(axis=0)
        dist_profiles[ds_short] = {
            f"Rank {r}": mean_dists[i] for i, r in enumerate(ranks_label)
        }

df_dists = pd.DataFrame(dist_profiles).T
df_dists.style.format("{:.4f}").set_caption("Mean Ground Truth Distances at Selected Neighbor Ranks")

In [ ]:
plt.figure(figsize=(12, 7))
for ds_name, profile in dist_profiles.items():
    y_vals = [profile[f"Rank {r}"] for r in ranks_label]
    plt.plot(ranks_label, y_vals, marker='o', label=ds_name, linewidth=2.5, markersize=8)

plt.title("Growth of Nearest Neighbor Distance by Rank", fontsize=14, fontweight='bold')
plt.xlabel("Neighbor Rank (Index 1 to 100)")
plt.ylabel("Average Distance Value")
plt.xscale('log')
plt.xticks(ranks_label, ranks_label)
plt.legend(title="Dataset", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4. Hubness Analysis (Query Concentration)

**Hubness** is a phenomenon in high-dimensional spaces where a small number of training vectors appear as the nearest neighbors for a disproportionately large number of query vectors. 
- If a single point is the nearest neighbor to dozens of queries, it acts as a "hub".
- Severe hubness can cause performance degradation or skew recall in graph-based search index models like HNSW.

Let's measure how many unique points are in the top-1 neighbor and top-10 neighbors list across all 1,000 queries, and find the maximum query frequency of a single training point.

In [ ]:
hubness_stats = []
for f_name in files:
    f_path = os.path.join(dataset_dir, f_name)
    ds_short = f_name.replace("-public.hdf5", "")
    with h5py.File(f_path, 'r') as f:
        neighbors = f['neighbors'][:]
        
        # Top-1 nearest neighbors
        unique1, counts1 = np.unique(neighbors[:, 0], return_counts=True)
        # Top-10 nearest neighbors
        flat_top10 = neighbors[:, :10].flatten()
        unique10, counts10 = np.unique(flat_top10, return_counts=True)
        
        hubness_stats.append({
            "Dataset": ds_short,
            "Unique 1st NNs (out of 1,000 queries)": len(unique1),
            "Max occurrences of a single 1st NN": np.max(counts1),
            "Unique Top-10 NNs (out of 10,000 entries)": len(unique10),
            "Max occurrences of a single Top-10 NN": np.max(counts10)
        })

df_hub = pd.DataFrame(hubness_stats)
df_hub.style.format({
    "Unique 1st NNs (out of 1,000 queries)": "{:,}",
    "Max occurrences of a single 1st NN": "{:,}",
    "Unique Top-10 NNs (out of 10,000 entries)": "{:,}",
    "Max occurrences of a single Top-10 NN": "{:,}"
}).set_caption("Hubness / Query Concentration Analysis")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Unique 1st neighbors (lower counts suggest higher concentration / hubness)
sns.barplot(data=df_hub, x="Dataset", y="Unique 1st NNs (out of 1,000 queries)", ax=axes[0], palette="Blues_r", hue="Dataset", legend=False)
axes[0].set_title("Number of Unique 1st Neighbors Found\n(Higher = More uniform neighborhood structure)", fontsize=12, fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha="right")
axes[0].set_ylabel("Unique Count")

# Right: Max frequency of a single point in top 10
sns.barplot(data=df_hub, x="Dataset", y="Max occurrences of a single Top-10 NN", ax=axes[1], palette="Reds", hue="Dataset", legend=False)
axes[1].set_title("Max Frequency of a Train Point in Top-10 lists\n(Higher = Presence of a strong 'hub' vector)", fontsize=12, fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha="right")
axes[1].set_ylabel("Max Query Hits")

plt.tight_layout()
plt.show()

## 5. Search Contrast & Discriminability

**Search Contrast** measures how much closer nearest neighbors are compared to typical/random vectors in the dataset. 
We calculate the **Contrast Ratio**:
$$\text{Contrast Ratio} = \frac{\text{Mean Distance to 1st Nearest Neighbor}}{\text{Mean Distance to Random Train Vectors}}$$
- A **lower ratio** (e.g. 0.2) means the nearest neighbors are highly distinct and stand out from the rest of the vector cloud.
- A **ratio close to 1.0** indicates that the nearest neighbors are almost as far away as any random vector, which is typical of extreme high dimensions ("loss of contrast"). This makes spatial partitioning extremely difficult.

In [ ]:
contrast_stats = []
for f_name in files:
    f_path = os.path.join(dataset_dir, f_name)
    ds_short = f_name.replace("-public.hdf5", "")
    with h5py.File(f_path, 'r') as f:
        train = f['train']
        test = f['test'][:]
        distances = f['distances'][:, 0]  # Ground truth 1st nearest neighbor distance
        neighbors = f['neighbors'][:, 0]
        
        # Randomly sample 200 training vectors to compute average global distance
        n_train = train.shape[0]
        np.random.seed(42)
        rand_indices = sorted(np.random.choice(n_train, min(n_train, 200), replace=False))
        train_sample = train[rand_indices]
        
        # Pairwise differences (1000, 200, dim)
        diffs = test[:, np.newaxis, :] - train_sample[np.newaxis, :, :]
        
        # Detect whether the ground truth distances are using L2 or squared L2
        # We check distance to true 1st neighbor for first 10 queries
        sample_nns = train[sorted(neighbors[:10])]
        # Map sorted indices back
        sort_map = {idx: i for i, idx in enumerate(sorted(neighbors[:10]))}
        sample_nns_ordered = np.array([sample_nns[sort_map[idx]] for idx in neighbors[:10]])
        
        sq_l2_computed = np.sum((test[:10] - sample_nns_ordered)**2, axis=1)
        
        is_squared = np.allclose(distances[:10], sq_l2_computed, atol=1e-2)
        
        if is_squared:
            metric_name = "Squared L2"
            global_dists = np.sum(diffs**2, axis=2)
        else:
            metric_name = "L2"
            global_dists = np.linalg.norm(diffs, axis=2)
            
        mean_nn_dist = np.mean(distances)
        mean_global_dist = np.mean(global_dists)
        contrast_ratio = mean_nn_dist / mean_global_dist
        
        contrast_stats.append({
            "Dataset": ds_short,
            "Metric Detected": metric_name,
            "Mean 1st NN Dist": mean_nn_dist,
            "Mean Global Dist": mean_global_dist,
            "Contrast Ratio": contrast_ratio
        })

df_contrast = pd.DataFrame(contrast_stats)
df_contrast.style.format({
    "Mean 1st NN Dist": "{:.4f}",
    "Mean Global Dist": "{:.4f}",
    "Contrast Ratio": "{:.4f}"
}).set_caption("Search Contrast Summary")

In [ ]:
plt.figure(figsize=(11, 6))
sns.barplot(data=df_contrast, x="Dataset", y="Contrast Ratio", palette="coolwarm", hue="Dataset", legend=False)
plt.title("Search Contrast Ratio (Mean NN Dist / Mean Global Dist)\n(Lower = Nearest Neighbors are more distinct from the database background)", fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha="right")
plt.ylabel("Contrast Ratio")
plt.axhline(1.0, color='red', linestyle='--', alpha=0.5, label="No Contrast (Ratio = 1.0)")
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 6. Key Conclusions & Implications for Index Tuning

Based on these empirical findings, we can derive the following design guidelines for tuning ANN algorithms (like HNSW, LSH simple vs optimized) on these specific scenarios:

1. **Normalization Aware Indexes**:
   - **Normalized group**: `gooaq-distilroberta`, `imagenet-clip`, `landmark-nomic`, `simplewiki-openai`, and `yahoo-minilm` are unit-sphere normalized. This means we can replace expensive $L_2$ distance calculations with fast dot products ($u \cdot v$) or cosine distance implementations to speed up queries.
   - **Non-normalized group**: `agnews-mxbai` and `celeba-resnet` are **not** normalized. Standard dot-product indexes will fail to yield correct nearest neighbors; Euclidean ($L_2$) distance indexes are strictly required.

2. **Dimensionality & Memory Footprints**:
   - `simplewiki-openai` has a dimension of **3,072**. A single float vector takes $12\text{ KB}$ of RAM. Storing the whole training set in raw float32 requires $2.97\text{ GB}$ of RAM. For large vectors, **Scalar Quantization (SQ8)** or **Product Quantization (PQ)** will be extremely beneficial to fit indices in memory and speed up distance calculations via INT8 arithmetic.
   - `yahoo-minilm` has a dimension of **384** and size of 676,305. The raw vector footprint is small, which allows fitting easily into cache and RAM.

3. **Dataset Discriminability (Contrast)**:
   - `landmark-nomic` has an extremely low contrast ratio of **0.268**, which means nearest neighbors are very close and highly distinct. This dataset is ideal for graph-based indices like HNSW, which will easily navigate towards the local neighbor hub.
   - `simplewiki-openai` has a contrast ratio of **0.825** and a high dimension of 3,072. The nearest neighbors are almost as far as random database points. This makes tree-based partitioning and graph indexing significantly harder because local routing decisions have very small distance deltas. Graph construction parameters (such as $M$ and $ef\_construction$ in HNSW) will likely need to be higher to maintain acceptable recall.